<a href="https://colab.research.google.com/github/hamnakhan11/hamna/blob/main/create_Dashboard_with_plotly_ans_Dash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
#question 2
!pip install dash
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd

url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/historical_automobile_sales.csv'
df = pd.read_csv(url)

app = dash.Dash(__name__)
app.title = "Automobile Sales Statistics Dashboard"

app.layout = html.Div([
    html.H1("Automobile Sales Statistics Dashboard", style={'textAlign': 'center'}),

    html.Div([
        html.Label("Select Statistics:"),
        dcc.Dropdown(
            id='dropdown-statistics',
            options=[
                {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
                {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
            ],
            value='Yearly Statistics',
            placeholder='Select a report type'
        )
    ]),

    html.Div([
        html.Label("Select Year:"),
        dcc.Dropdown(
            id='select-year',
            options=[{'label': y, 'value': y} for y in sorted(df['Year'].unique())],
            value=int(df['Year'].min())
        )
    ]),

    html.Div(id='output-container', className='chart-grid')
])

# --- callback 1: enable/disable the year dropdown ---
@app.callback(
    Output('select-year', 'disabled'),
    Input('dropdown-statistics', 'value')
)
def toggle_year_dropdown(selected_stat):
    return selected_stat == 'Recession Period Statistics'


# --- callback 2: update the charts ---
@app.callback(
    Output('output-container', 'children'),
    [Input('dropdown-statistics', 'value'), Input('select-year', 'value')]
)
def update_output(selected_stat, selected_year):
    if selected_stat == 'Recession Period Statistics':
        rec = df[df['Recession'] == 1]

        chart1 = dcc.Graph(figure=px.line(
            rec.groupby('Year')['Automobile_Sales'].mean().reset_index(),
            x='Year', y='Automobile_Sales', title='Sales Trend during Recession'))

        chart2 = dcc.Graph(figure=px.bar(
            rec.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index(),
            x='Vehicle_Type', y='Automobile_Sales', title='Avg Sales by Vehicle Type'))

        chart3 = dcc.Graph(figure=px.pie(
            rec.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index(),
            names='Vehicle_Type', values='Advertising_Expenditure',
            title='Ad Expenditure Share by Vehicle Type (Recession)'))

        unemp = rec.groupby(['unemployment_rate', 'Vehicle_Type'])['Automobile_Sales'].mean().reset_index()
        chart4 = dcc.Graph(figure=px.bar(
            unemp, x='unemployment_rate', y='Automobile_Sales', color='Vehicle_Type',
            title='Effect of Unemployment Rate on Sales by Vehicle Type'))

        return [html.Div(className='chart-grid', children=[chart1, chart2, chart3, chart4],
                          style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr'})]

    else:
        yr = df[df['Year'] == selected_year]

        chart1 = dcc.Graph(figure=px.line(
            df.groupby('Year')['Automobile_Sales'].mean().reset_index(),
            x='Year', y='Automobile_Sales', title='Yearly Sales Trend'))

        chart2 = dcc.Graph(figure=px.line(
            yr.groupby('Month')['Automobile_Sales'].mean().reset_index(),
            x='Month', y='Automobile_Sales', title=f'Total Monthly Sales in {selected_year}'))

        chart3 = dcc.Graph(figure=px.bar(
            yr.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index(),
            x='Vehicle_Type', y='Automobile_Sales', title=f'Avg Vehicles Sold by Type in {selected_year}'))

        chart4 = dcc.Graph(figure=px.pie(
            yr.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index(),
            names='Vehicle_Type', values='Advertising_Expenditure',
            title=f'Ad Expenditure by Vehicle Type in {selected_year}'))

        return [html.Div(className='chart-grid', children=[chart1, chart2, chart3, chart4],
                          style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr'})]


if __name__ == '__main__':
    app.run(debug=False)

<IPython.core.display.Javascript object>